In [ ]:
import os
import json
import csv
import time
from datetime import datetime

import pandas as pd
from openai import OpenAI


# -----------------------
# CONFIG
# -----------------------
API_KEY = os.environ.get("OPENAI_API_KEY", "OPENAI_API_KEY")
MODEL_ID = "gpt-5.2"

DATA_DIR = "Data"
PROMPT_DIR = "Prompts"
OUTPUT_ROOT = "output_batch_files"

INPUT_BATCH_FILE = "input_batch_files/batch_requests.jsonl"
RAW_OUTPUT_FILE = "batch_raw_results/batch_raw_results.jsonl"

# Add as many as you want (must have a corresponding .txt file in PROMPT_DIR)
PROMPTS = [
    "CoT",
    "vanilla",
    "multi_step",
    "top_k",
    "formatted",
    "repetitions"
]

MULTIPLE_CHOICE_TASKS = {"gpqa", "openbookqa"}

POLL_SECONDS = 60

# You can keep these exactly as in your Gemini script
PROMPT_SUFFIX = "Now, please answer this question and provide your confidence level."
COT_SUFFIX = " Let's think it step by step."

client = OpenAI(api_key=API_KEY)


# -----------------------
# HELPERS
# -----------------------
def safe_mkdir(path: str):
    os.makedirs(path, exist_ok=True)


def list_tasks(data_dir: str):
    tasks = []
    for fn in os.listdir(data_dir):
        if fn.endswith(".csv") and os.path.isfile(os.path.join(data_dir, fn)):
            tasks.append(os.path.splitext(fn)[0])
    return sorted(tasks)


def prompt_template_path(prompt_name: str, task_name: str) -> str:
    """
    CoT has a special wording for multiple-choice datasets.
    """
    if prompt_name == "CoT" and task_name in MULTIPLE_CHOICE_TASKS:
        return os.path.join(PROMPT_DIR, "multiple_choice", "CoT.txt")
    return os.path.join(PROMPT_DIR, f"{prompt_name}.txt")


def load_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def build_user_prompt(prompt_name: str, row: dict, prompt_prefix: str) -> str:
    """
    Mirrors your Gemini branching logic.
    """
    q = row.get("question", "")

    if prompt_name == "self_probing":
        prev_response = row.get("prev_response", "")
        # Same as your example:
        # formatted_prompt = f"Question: {row['question']}\nPossible Answer: {prev_response}\n{prompt_prefix}"
        return f"Question: {q}\nPossible Answer: {prev_response}\n{prompt_prefix}"

    if prompt_name == "vanilla":
        # formatted_prompt = f"{prompt_prefix}\nQuestion: {row['question']}\n{prompt_suffix}"
        return f"{prompt_prefix}\nQuestion: {q}\n{PROMPT_SUFFIX}"

    if prompt_name == "CoT":
        # formatted_prompt = f"{prompt_prefix}\nQuestion: {row['question']}\n{prompt_suffix}{cot_suffix}"
        return f"{prompt_prefix}\nQuestion: {q}\n{PROMPT_SUFFIX}{COT_SUFFIX}"

    if prompt_name in ("repetitions", "formatted"):
        # formatted_prompt = f"Question: {row['question']}\n{prompt_prefix}"
        return f"Question: {q}\n{prompt_prefix}"

    # Default:
    # formatted_prompt = f"{prompt_prefix}\nQuestion: {row['question']}"
    return f"{prompt_prefix}\nQuestion: {q}"


def extract_openai_batch_message(result_obj: dict) -> str:
    """
    OpenAI batch output lines typically:
    {
      "custom_id": "...",
      "response": { "status_code": 200, "body": {choices:[...]} }
    }
    """
    try:
        body = result_obj["response"]["body"]
        return body["choices"][0]["message"]["content"]
    except Exception:
        err = result_obj.get("error") or result_obj.get("response", {}).get("body", {}).get("error")
        if err:
            return f"[ERROR] {err}"
        return ""


# -----------------------
# STEP 1: BUILD REQUESTS.JSONL
# -----------------------
def build_requests_jsonl(output_jsonl_path: str) -> int:
    count = 0
    tasks = list_tasks(DATA_DIR)

    with open(output_jsonl_path, "w", encoding="utf-8") as out:
        for task in tasks:
            df = pd.read_csv(os.path.join(DATA_DIR, f"{task}.csv"))

            for prompt_name in PROMPTS:
                template_file = prompt_template_path(prompt_name, task)
                prompt_prefix = load_text(template_file)

                for _, r in df.iterrows():
                    row = r.to_dict()

                    instance_id = row["instance_id"]
                    user_text = build_user_prompt(prompt_name, row, prompt_prefix)

                    custom_id = f"{task}|{prompt_name}|{instance_id}"

                    req = {
                        "custom_id": custom_id,
                        "method": "POST",
                        "url": "/v1/chat/completions",
                        "body": {
                            "model": MODEL_ID,
                            "temperature": 0,
                            "max_completion_tokens": 1000,
                            "messages": [
                                {"role": "user", "content": user_text}
                            ],
                        },
                    }

                    out.write(json.dumps(req, ensure_ascii=False) + "\n")
                    count += 1

    return count


# -----------------------
# STEP 2: SUBMIT BATCH
# -----------------------
def submit_batch(input_jsonl_path: str) -> str:
    uploaded = client.files.create(
        file=open(input_jsonl_path, "rb"),
        purpose="batch",
    )

    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={"display_name": "bigbench-eval"},
    )

    return batch.id


# -----------------------
# STEP 3: POLL + DOWNLOAD RESULTS
# -----------------------
def wait_for_batch_and_download(batch_id: str, raw_output_path: str) -> str:
    while True:
        batch = client.batches.retrieve(batch_id)
        state = batch.status
        now = datetime.now().strftime("%H:%M:%S")
        print(f"[{now}] Batch {batch_id} status: {state}")

        if state == "completed":
            break
        if state in ("failed", "cancelled", "expired"):
            raise RuntimeError(f"Batch ended with status={state}. batch_id={batch_id}")

        time.sleep(POLL_SECONDS)

    if not batch.output_file_id:
        raise RuntimeError("Batch completed but output_file_id is missing.")

    content = client.files.content(batch.output_file_id)
    with open(raw_output_path, "wb") as f:
        f.write(content.read())

    return raw_output_path


# -----------------------
# STEP 4: WRITE Outputs/<task>/<prompt>.csv
# -----------------------
def write_final_csvs(raw_output_path: str):
    # Map (task, instance_id) -> (question, target)
    task_maps = {}
    for task in list_tasks(DATA_DIR):
        df = pd.read_csv(os.path.join(DATA_DIR, f"{task}.csv"))
        m = {}
        for _, r in df.iterrows():
            m[str(r["instance_id"])] = (r["question"], r["target"])
        task_maps[task] = m

    writers = {}

    def get_writer(task: str, prompt: str):
        key = (task, prompt)
        if key in writers:
            return writers[key]

        out_dir = os.path.join(OUTPUT_ROOT, task)
        safe_mkdir(out_dir)
        out_path = os.path.join(out_dir, f"{prompt}.csv")

        fh = open(out_path, "w", newline="", encoding="utf-8")
        w = csv.writer(fh)
        w.writerow(["instance_id", "question", "target", "response"])
        writers[key] = (fh, w)
        return fh, w

    with open(raw_output_path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            custom_id = obj.get("custom_id", "")
            if "|" not in custom_id:
                continue

            task, prompt, instance_id = custom_id.split("|", 2)
            instance_id = str(instance_id)

            question, target = task_maps.get(task, {}).get(instance_id, ("", ""))
            response_text = extract_openai_batch_message(obj)

            _, w = get_writer(task, prompt)
            w.writerow([instance_id, question, target, response_text])

    for fh, _ in writers.values():
        fh.close()

In [ ]:
# -----------------------
# RUN
# -----------------------
if __name__ == "__main__":
    print(f"Building JSONL: {INPUT_BATCH_FILE}")
    n = build_requests_jsonl(INPUT_BATCH_FILE)
    print(f"✅ Wrote {n} requests.")

    print("Submitting OpenAI batch...")
    batch_id = submit_batch(INPUT_BATCH_FILE)
    print(f"✅ Batch submitted. batch_id={batch_id}")

    print("Polling until completed, then downloading raw results...")
    wait_for_batch_and_download(batch_id, RAW_OUTPUT_FILE)
    print(f"✅ Raw results saved to {RAW_OUTPUT_FILE}")

    print("Writing final per-task/per-prompt CSVs...")
    write_final_csvs(RAW_OUTPUT_FILE)
    print(f"✅ Done. Outputs written under: {OUTPUT_ROOT}")


In [ ]:
RAW_OUTPUT_FILE = "batch_raw_results.jsonl"

if __name__ == "__main__":

    print("Writing final per-task/per-prompt CSVs...")
    write_final_csvs(RAW_OUTPUT_FILE)
    print(f"✅ Done. Outputs written under: {OUTPUT_ROOT}")